# 🛡️ FraudLens AI: Intelligent Financial Fraud Detection
### Complete Machine Learning Lifecycle & PBL Evaluation Rubric Implementation
**Objective:** Detect fraudulent financial transactions using Supervised Classification and Unsupervised Anomaly Detection.

**PBL Rubric Mapping (100 Marks):**
1. Problem Definition & Business Understanding (10 marks)
2. Dataset & Data Understanding (10 marks)
3. Data Preprocessing & Cleaning (10 marks)
4. Exploratory Data Analysis (EDA) (10 marks)
5. Feature Engineering & Selection (10 marks)
6. Model Selection & Implementation (15 marks)
7. Model Evaluation & Validation (15 marks)
8. Business Insights & Recommendations (10 marks)
9. Deployment / Usable Interface (5 marks)
10. Documentation & Presentation (5 marks)

## 1. Setup & Environment Imports
Importing core data manipulation, visualization, modeling, and evaluation libraries.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.sans-serif': 'Arial', 'font.size': 11})

# Verify working directory and import source modules
project_root = os.path.abspath('..') if os.path.exists('../src') else os.path.abspath('.')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project Root: {project_root}")

## 2. Dataset Acquisition & Understanding (10 Marks)
We utilize the benchmark financial transaction stream modeled after **Kaggle PaySim** (`ntnu-testimon/paysim1`).
- Explores transaction types (`CASH_IN`, `CASH_OUT`, `DEBIT`, `PAYMENT`, `TRANSFER`)
- Audits severe class imbalance (0.5% fraud vs 99.5% legitimate)

In [ ]:
from src.data_generator import generate_financial_data

data_path = os.path.join(project_root, 'data/raw/financial_transactions.csv')
if not os.path.exists(data_path):
    generate_financial_data(output_path=data_path)

df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
display(df.head(5))
print("\n--- Data Info ---")
df.info()

## 3. Exploratory Data Analysis (EDA) (10 Marks)
Visualizing class skew, transaction types, log-amount distributions, and temporal cyber-attack windows.

In [ ]:
# 1. Class Imbalance Breakdown
fig, ax = plt.subplots(figsize=(7, 4.5))
counts = df['is_fraud'].value_counts()
bars = ax.bar(['Legitimate (0)', 'Fraudulent (1)'], counts.values, color=['#2b5c8f', '#d9534f'], width=0.45)
ax.set_yscale('log')
ax.set_title("Transaction Class Distribution (Log Scale)", fontsize=13, weight='bold')
for bar in bars:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, y * 1.15, f"{y:,} ({y/len(df)*100:.2f}%)", ha='center', weight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Fraud Vulnerability by Transaction Type
type_summary = df.groupby('type')['is_fraud'].agg(['count', 'sum']).reset_index()
type_summary['fraud_pct'] = (type_summary['sum'] / type_summary['count']) * 100.0

plt.figure(figsize=(8, 4.5))
sns.barplot(data=type_summary, x='type', y='fraud_pct', hue='type', palette='Reds_r', legend=False)
plt.title("Fraud Occurrence Rate (%) by Transaction Type", fontsize=13, weight='bold')
plt.ylabel("Fraud Rate (%)")
plt.xlabel("Transaction Type")
plt.show()

## 4. Feature Engineering (10 Marks)
Domain-engineered features to capture fraud mechanics:
1. `error_balance_orig`: Originator balance discrepancy
2. `orig_account_drained`: Account liquidation flag
3. `dest_zero_initial` & `dest_zero_after`: Mule account behavior
4. `amount_to_orig_ratio`: Relative financial impact
5. `hour_of_day`, `is_night_hours`: Temporal anomaly patterns

In [ ]:
from src.feature_engineering import engineer_features, get_feature_columns

df_eng = engineer_features(df)
print(f"Total features after engineering: {df_eng.shape[1]}")
display(df_eng[['amount', 'error_balance_orig', 'orig_account_drained', 'dest_zero_initial', 'is_night_hours', 'is_fraud']].head(5))

## 5. Preprocessing & Data Leakage Prevention (10 Marks)
We split into Train and Test partitions **prior** to scaling using Stratified splitting, ensuring no leakage.

In [ ]:
from src.preprocessing import FraudDataPipeline

pipeline = FraudDataPipeline(target_col='is_fraud')
X_train, X_test, y_train, y_test = pipeline.split_and_prepare(df, test_size=0.20, random_state=42)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 6. Model Training & Comparison (15 Marks)
We implement a multi-model benchmarking pipeline:
1. Cost-Sensitive Logistic Regression
2. Balanced Random Forest Classifier
3. Histogram Gradient Boosting Classifier
4. XGBoost Classifier
5. Isolation Forest (Unsupervised Anomaly Detector)

In [ ]:
from src.models import FraudLensModelSuite

suite = FraudLensModelSuite(random_state=42)
suite.initialize_models(scale_pos_weight=150.0)
suite.train_supervised(X_train, y_train, X_test, y_test)
suite.train_unsupervised(X_train, y_train, X_test, y_test)

## 7. Model Evaluation & ROC / PR Benchmarks (15 Marks)
Precision-Recall AUC is the gold standard for severe class imbalance.

In [ ]:
metrics_df = pd.DataFrame(suite.model_metrics).T
display(metrics_df[['pr_auc', 'roc_auc', 'recall', 'precision', 'f1', 'fpr']])

## 8. Business Impact & Financial ROI Analysis (10 Marks)
Converting machine learning outputs into real-world dollar savings and risk mitigation.

In [ ]:
from src.evaluate import evaluate_cost_benefit, find_optimal_threshold

best_clf = suite.best_classifier
y_prob = best_clf.predict_proba(X_test)[:, 1]
test_amounts = df.loc[X_test.index, 'amount'].values if hasattr(X_test, 'index') else np.ones(len(y_test)) * 1000

best_thresh, _ = find_optimal_threshold(y_test, y_prob)
financial_results = evaluate_cost_benefit(y_test, (y_prob >= best_thresh).astype(int), test_amounts)

print(f"Optimal Decision Threshold: {best_thresh:.3f}")
for k, v in financial_results.items():
    print(f"{k}: {v}")

## 9. Conclusion & Deployment
The trained models and preprocessing pipelines are serialized in `models/`.
To launch the interactive web dashboard:
```bash
streamlit run app.py
```